# Kaggle evaluation run — finalproject_cpe027

Runs the Tier 1 (ACL6060 + seeded DEMAND noise) and Tier 2 (unmodified M3AV) ASR evaluation
on Kaggle (GPU) using preprocessed local data. No preprocessing is redone here.

- Tiers are executed and reported separately; they are never pooled.
- Checkpoints are retained per tier; this notebook starts fresh (`--no-resume`).
- Whisper: `base`, `cuda`, `float16`, `en` (Tier 1). DSP/decoding settings come from the runner.
- Data mounts are discovered recursively (nested under /kaggle/input/datasets/<user>/...).


In [40]:
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

# ---- resolve project code (attached dataset first, git fallback) ---------
REPO_DIR = None
for p in Path("/kaggle/input").rglob("backend"):
    if (p / "app").is_dir():
        REPO_DIR = p.parent
        break
REPO_DIR = REPO_DIR or Path("/kaggle/working/finalproject_cpe027")
BACKEND_DIR = REPO_DIR / "backend"

REPO_GIT_URL = os.environ.get(
    "REPO_GIT_URL", "https://github.com/leon-pscl/finalproject_cpe027.git"
).strip()
if not REPO_DIR.is_dir():
    print("Cloning project repository ...")
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_GIT_URL, str(REPO_DIR)], check=True
    )
if not (BACKEND_DIR / "app").is_dir():
    raise FileNotFoundError(
        "Backend directory not found. Attach a dataset containing backend/app "
        "or make the repository public and ensure git can clone it."
    )

# ---- resolve data mounts recursively (nested under datasets/<user>) ----
INPUT = Path("/kaggle/input")
SEARCH_ROOTS = [INPUT, *[p for p in INPUT.glob("datasets/*") if p.is_dir()],
                 *[p for p in INPUT.glob("datasets/*/*") if p.is_dir()]]

acl_hits = []
for p in SEARCH_ROOTS:
    cand = (p / "dev" / "full_wavs")
    if cand.is_dir():
        acl_hits.append(p)
acl_hits = list(dict.fromkeys(acl_hits))
if not acl_hits:
    listing = sorted(str(p.name) for p in INPUT.iterdir()) if INPUT.is_dir() else []
    raise FileNotFoundError(
        "ACL6060 data not found. Scanned roots: " + ", ".join(str(p) for p in SEARCH_ROOTS)
        + ". Top-level entries: " + ", ".join(listing)
    )

demand_hits = []
for p in SEARCH_ROOTS:
    for cand in (p, p / "DEMAND"):
        if (cand / "DKITCHEN_16k").is_dir() and next(cand.rglob("*.wav"), None) is not None:
            demand_hits.append(cand)
demand_hits = list(dict.fromkeys(demand_hits))
if not demand_hits:
    listing = sorted(str(p.name) for p in INPUT.iterdir()) if INPUT.is_dir() else []
    raise FileNotFoundError(
        "DEMAND data not found. Scanned roots: " + ", ".join(str(p) for p in SEARCH_ROOTS)
        + ". Top-level entries: " + ", ".join(listing)
    )

m3av_raw = sorted({str(p / "M3AV" / "extracted")
                for p in SEARCH_ROOTS
                if (p / "M3AV" / "extracted").is_dir()})
m3av_zips = sorted({z for p in SEARCH_ROOTS for z in p.rglob("*.zip") if "m3av" in z.name.lower()})

ACL_ROOT = acl_hits[0]
DEMAND_ROOT = demand_hits[0]

os.environ["REPO_DIR"] = str(REPO_DIR)
os.environ["BACKEND_DIR"] = str(BACKEND_DIR)
os.environ["ACL_ROOT"] = str(ACL_ROOT)
os.environ["DEMAND_ROOT"] = str(DEMAND_ROOT)
M3AV_ZIPS = [str(z) for z in m3av_zips]
os.environ["M3AV_ZIPS"] = chr(10).join(M3AV_ZIPS)
os.environ["M3AV_CLIP_ROOT"] = "\n".join(m3av_raw)

print("ACL root:", ACL_ROOT)
print("DEMAND root:", DEMAND_ROOT)
print("M3AV raw roots:", m3av_raw)
print("M3AV zips (lazy-extract):", [z.name for z in m3av_zips], flush=True)
!nvidia-smi


ACL root: /kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-acl6060
DEMAND root: /kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-demand/DEMAND
M3AV raw roots: ['/kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-m3av-split-0/M3AV/extracted', '/kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-m3av-split-1/M3AV/extracted', '/kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-m3av-split-2/M3AV/extracted', '/kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-m3av-split-3/M3AV/extracted']
M3AV zips (lazy-extract): []
Fri Nov  6 06:53:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf 

In [41]:
!df -h

Filesystem                         Size  Used Avail Use% Mounted on
overlay                            8.0T  7.1T  967G  89% /
tmpfs                               64M     0   64M   0% /dev
shm                                 14G  4.0K   14G   1% /dev/shm
/dev/sda1                          122G  6.4G  116G   6% /opt/bin
/dev/loop2                          20G  604M   19G   4% /kaggle/lib
kkb-production-mirror-us-central1  1.0P     0  1.0P   0% /kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-acl6060
kkb-production-mirror-us-central1  1.0P     0  1.0P   0% /kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-demand
/dev/mapper/snap                   8.0T  7.1T  967G  89% /etc/hosts
tmpfs                              4.0K     0  4.0K   0% /proc/acpi
kkb-production-mirror-us-central1  1.0P     0  1.0P   0% /kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-m3av-split-2
kkb-production-mirror-us-central1  1.0P     0  1.0P   0% /kaggle/input/datasets/leonpascual1

In [42]:
from pathlib import Path

print("Input directory tree (up to depth 3):")
for p in sorted(Path("/kaggle/input").rglob("*")):
    if not p.is_dir():
        continue
    rel = p.relative_to("/kaggle/input")
    if len(rel.parts) <= 3:
        print("  " * (len(rel.parts) - 1) + rel.name + "/")


Input directory tree (up to depth 3):
datasets/
  leonpascual1/
    finalproject-cpe027-eval-acl6060/
    finalproject-cpe027-eval-demand/
    finalproject-cpe027-eval-m3av-split-0/
    finalproject-cpe027-eval-m3av-split-1/
    finalproject-cpe027-eval-m3av-split-2/
    finalproject-cpe027-eval-m3av-split-3/


In [43]:
!pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12==9.*

In [44]:
!pip install -q faster-whisper av scipy numpy rapidfuzz matplotlib pydantic-settings python-dotenv httpx pydantic

In [45]:
os.environ["ENV"] = "production"
os.environ["PROCESSING_MODE"] = "real"
# arm_a SNR fix: forced signal_snr_improvement_db=0.0 in eval_runner (see docs)
import os

os.environ["WHISPER_MODEL"] = "base"
os.environ["WHISPER_DEVICE"] = "cuda"
os.environ["WHISPER_COMPUTE_TYPE"] = "float16"
os.environ["WHISPER_THREADS"] = "4"
os.environ["WHISPER_LANGUAGE"] = "en"
os.environ["MODEL_CACHE_DIR"] = "/kaggle/working/models"
os.environ["EVAL_DATA_ROOT"] = os.environ.get("ACL_ROOT", "/kaggle/input/acl6060")
# arm_a SNR fix: forced signal_snr_improvement_db=0.0 in eval_runner (see docs)
# Attach M3AV split zip paths (Tier 2)
os.environ["M3AV_ZIPS"] = "E:\\Datasets\\kaggle_datasets\\m3av_0\\m3av_0.zip\nE:\\Datasets\\kaggle_datasets\\m3av_1\\m3av_1.zip\nE:\\Datasets\\kaggle_datasets\\m3av_2\\m3av_2.zip\nE:\\Datasets\\kaggle_datasets\\m3av_3\\m3av_3.zip"

# ---- Verify CUDA / CTranslate2 capability (GPU fallback check) ----
try:
    import ctranslate2
    print("ctranslate2:", ctranslate2.__version__)
    print("CUDA compute types:", ctranslate2.get_supported_compute_types("cuda"))
    print("CPU compute types:", ctranslate2.get_supported_compute_types("cpu"))
except Exception as exc:
    print("CTranslate2 CUDA check failed:", type(exc).__name__, exc)
try:
    import torch
    print("torch CUDA available:", torch.cuda.is_available(), "count:", torch.cuda.device_count())
except Exception as exc:
    print("torch CUDA check failed:", type(exc).__name__, exc)

ctranslate2: 4.8.5
CUDA compute types: {'int8_float32', 'float16', 'int8', 'float32', 'int8_float16'}
CPU compute types: {'int8_float32', 'int16', 'int8', 'float32'}
torch CUDA available: True count: 2


In [46]:
import subprocess
import sys
from importlib.metadata import version


def av_ok() -> bool:
    """Probe in a fresh interpreter so we see the currently installed av."""
    probe = subprocess.run(
        [sys.executable, "-c",
         "import av; av.open('/nonexistent.wav', metadata_errors='ignore')"],
        capture_output=True, text=True,
    )
    last = (probe.stderr.strip().splitlines() or [""])[-1]
    return "unexpected keyword argument 'metadata_errors'" not in last


if not av_ok():
    print(f"av {version('av')} is incompatible with faster-whisper "
          f"{version('faster-whisper')}; trying older prebuilt wheels ...")
    fixed = False
    # Newest prebuilt wheel below each ceiling; never compile from source.
    for ceiling in ["19", "17", "16", "15", "14", "13", "12"]:
        spec = f"av<{ceiling}"
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "--only-binary=:all:", spec],
            capture_output=True, text=True,
        )
        if r.returncode != 0:
            print(f"  {spec}: no usable wheel")
            continue
        ok = av_ok()
        print(f"  {spec}: installed av {version('av')} -> {'OK' if ok else 'still incompatible'}")
        if ok:
            fixed = True
            break
    if not fixed:
        raise RuntimeError(
            "No prebuilt av wheel works with this faster-whisper. Options: "
            "(1) `!python -m pip install -q 'faster-whisper<1.1'` and re-run this cell, or "
            "(2) paste this output so we can pick another pin."
        )
print("av/faster-whisper compatibility: OK\n")

for pkg in ["ctranslate2", "faster-whisper", "av", "scipy", "numpy", "pandas",
            "rapidfuzz", "matplotlib", "pydantic", "pydantic_settings"]:
    try:
        print(f"{pkg}: {version(pkg)}")
    except Exception:
        print(f"{pkg}: unknown")

try:
    import torch
    print("torch:", torch.__version__)
except Exception:
    print("torch: not installed")

av/faster-whisper compatibility: OK

ctranslate2: 4.8.5
faster-whisper: 1.2.1
av: 18.1.0
scipy: 1.16.3
numpy: 2.1.3
pandas: 2.3.3
rapidfuzz: 3.14.6
matplotlib: 3.10.0
pydantic: 2.14.1
pydantic_settings: 2.16.2
torch: 2.11.0+cu130


## Tier 1 smoke test


In [117]:
!pip install -q faster-whisper "av==18.1.0" scipy numpy rapidfuzz matplotlib pydantic-settings python-dotenv httpx pydantic
from importlib.metadata import version
print("av:", version("av"), "| faster-whisper:", version("faster-whisper"))

av: 18.1.0 | faster-whisper: 1.2.1


In [118]:
import glob
import os
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "nvidia-cublas-cu12", "nvidia-cudnn-cu12"], check=True)

# Locate the installed libs and prepend them to LD_LIBRARY_PATH.
# Shell commands (!python ...) inherit this, so the runner subprocess sees it.
site = next(p for p in sys.path if p.endswith("dist-packages") and os.path.isdir(os.path.join(p, "nvidia")))
lib_dirs = glob.glob(os.path.join(site, "nvidia", "cublas", "lib")) + \
           glob.glob(os.path.join(site, "nvidia", "cudnn", "lib"))
print("lib dirs:", lib_dirs)

os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])

print(glob.glob(os.path.join(site, "nvidia", "cublas", "lib", "libcublas.so.12*")))

lib dirs: ['/usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib', '/usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib']
['/usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib/libcublas.so.12']


In [119]:
import os
from pathlib import Path

ACL_ROOT = os.environ.get("ACL_ROOT")
DEMAND_ROOT = os.environ.get("DEMAND_ROOT")
BACKEND_DIR = os.environ.get("BACKEND_DIR", "/kaggle/working/finalproject_cpe027/backend")

if not (ACL_ROOT and DEMAND_ROOT and Path(ACL_ROOT).is_dir() and Path(DEMAND_ROOT).is_dir()):
    raise RuntimeError("Attach ACL6060 and DEMAND datasets before the smoke test.")

os.chdir(BACKEND_DIR)
os.environ["WHISPER_DEVICE"] = "cuda"
os.environ["WHISPER_COMPUTE_TYPE"] = "float16"
!python -m app.eval_runner --tier 1 --limit 1 --split eval --conditions arm_a,hp --snrs 0 --noise demand --demand-envs DKITCHEN,DLIVING,DWASHING --data-root "$ACL_ROOT" --demand-root "$DEMAND_ROOT" --out /kaggle/working/eval_smoke_tier1


Loading ASR model...
Tier 1: 5 talks, SNRs=[0.0]
  2022.acl-long.111 | demand:DKITCHEN | SNR=0.0 | already complete; skipping
  2022.acl-long.111 | demand:DLIVING | SNR=0.0 | already complete; skipping
  2022.acl-long.111 | demand:DWASHING | SNR=0.0 | already complete; skipping
{
  "records": 6,
  "summaries": 1,
  "out": "/kaggle/working/eval_smoke_tier1"
}


In [120]:
import json
from pathlib import Path

t1 = json.loads(Path("/kaggle/working/eval_smoke_tier1/tier1_acl_results.json").read_text())
print("SMOKE tier1 records:", len(t1))
r = t1[0]
print(f"SMOKE WER: {r["wer"]:.2%} | RTF: {r["rtf"]:.3f} | seconds: {r["seconds"]:.1f}")


SMOKE tier1 records: 6
SMOKE WER: 26.69% | RTF: 0.054 | seconds: 38.8


## Full run  Tier 1 (ACL6060 + seeded DEMAND noise)

All six conditions, SNRs `none,0,5,10,20`, DEMAND envs `DKITCHEN,DLIVING,DWASHING`. Fresh run (`--no-resume`).

In [22]:
!grep -n -i -E "resume|checkpoint" /kaggle/working/finalproject_cpe027/backend/app/eval_runner.py | head -40

113:CHECKPOINT_SCHEMA_VERSION = 2
118:class CheckpointIncompatibleError(RuntimeError):
119:    """Raised when a checkpoint cannot be safely resumed for the current run."""
200:def _load_checkpoint(path: Path, expected_manifest: dict[str, Any] | None = None) -> dict[str, Any]:
202:        return {"schema_version": CHECKPOINT_SCHEMA_VERSION, "records": [], "run_manifest": expected_manifest}
207:        raise CheckpointIncompatibleError(f"incompatible checkpoint (unreadable): {exc}") from exc
209:        raise CheckpointIncompatibleError("incompatible checkpoint (missing records list)")
211:    if schema_version != CHECKPOINT_SCHEMA_VERSION:
212:        raise CheckpointIncompatibleError(
213:            f"incompatible checkpoint schema {schema_version}; "
214:            "start a new run with --no-resume or remove the checkpoint"
217:        raise CheckpointIncompatibleError("incompatible checkpoint manifest/fingerprint")
219:        # Existing checkpoint must be validated by its own mani

In [122]:
import os
from pathlib import Path

ACL_ROOT = os.environ.get("ACL_ROOT")
DEMAND_ROOT = os.environ.get("DEMAND_ROOT")
if not ACL_ROOT or not Path(ACL_ROOT).is_dir():
    raise RuntimeError("Attach ACL6060 data before Tier 1")
if not DEMAND_ROOT or not Path(DEMAND_ROOT).is_dir():
    raise RuntimeError("Attach DEMAND data before Tier 1")
os.chdir(os.environ.get("BACKEND_DIR", "/kaggle/working/finalproject_cpe027/backend"))
os.environ["WHISPER_DEVICE"] = "cuda"
os.environ["WHISPER_COMPUTE_TYPE"] = "float16"
!python -m app.eval_runner --tier 1 --split eval --conditions arm_a,hp,hp+sg,hp_ffmpeg,hp_wiener,hp_wiener_oa --snrs none,0,5,10,20 --noise demand --demand-envs DKITCHEN,DLIVING,DWASHING --data-root "$ACL_ROOT" --demand-root "$DEMAND_ROOT" --out /kaggle/working/batches/tier1_eval


Loading ASR model...
Tier 1: 5 talks, SNRs=[None, 0.0, 5.0, 10.0, 20.0]
  2022.acl-long.111 | none | SNR=None | already complete; skipping
  2022.acl-long.111 | demand:DKITCHEN | SNR=0.0 | already complete; skipping
  2022.acl-long.111 | demand:DKITCHEN | SNR=5.0 | already complete; skipping
  2022.acl-long.111 | demand:DKITCHEN | SNR=10.0 | already complete; skipping
  2022.acl-long.111 | demand:DKITCHEN | SNR=20.0 | already complete; skipping
  2022.acl-long.111 | none | SNR=None | already complete; skipping
  2022.acl-long.111 | demand:DLIVING | SNR=0 dB | arm_a | already complete; skipping
  2022.acl-long.111 | demand:DLIVING | SNR=0 dB | hp | already complete; skipping
  2022.acl-long.111 | demand:DLIVING | SNR=0 dB | hp+sg | already complete; skipping
  2022.acl-long.111 | demand:DLIVING | SNR=0 dB | hp_ffmpeg
    ASR segments: 1 (0.6-7.1s)
    ASR segments: 10 (46.0-48.0s)
    ASR segments: 20 (96.0-105.3s)
    ASR segments: 30 (159.3-162.3s)
    ASR segments: 40 (224.1-231.2s)


## Tier 2 smoke test


In [32]:
import os, shutil, zipfile
from pathlib import Path

BACKEND_DIR = os.environ.get("BACKEND_DIR", "/kaggle/working/finalproject_cpe027/backend")
M3AV_ROOT = os.environ.get("M3AV_CLIP_ROOT", "").splitlines()[0] if os.environ.get("M3AV_CLIP_ROOT", "").splitlines() else ""

if not (M3AV_ROOT and Path(M3AV_ROOT).is_dir()):
    zips = [z for z in os.environ.get("M3AV_ZIPS", "").splitlines() if z.strip()]
    if not zips:
        raise RuntimeError("Attach at least one M3AV split dataset before the smoke test.")
    stamp = Path("/kaggle/working/mounts/_smoke_m3av")
    if not (stamp / "M3AV" / "extracted").is_dir():
        if stamp.exists(): shutil.rmtree(stamp)
        stamp.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as z: z.extractall(stamp)
    M3AV_ROOT = str(stamp / "M3AV" / "extracted")

os.chdir(BACKEND_DIR)
os.environ["WHISPER_DEVICE"] = "cuda"
os.environ["WHISPER_COMPUTE_TYPE"] = "float16"
!python -m app.eval_runner --tier 2 --conditions arm_a,hp --snrs 0 --m3av-root "$M3AV_ROOT" --out /kaggle/working/eval_smoke_tier2 --limit 1 --no-resume


Loading ASR model...
Tier 2: 1 clips
  CHI-003EC | arm_a
  ASR segments: 1 (4.0-9.9s)
  ASR segments: 10 (36.3-40.1s)
  ASR segments: 20 (67.2-71.0s)
  ASR segments: 30 (97.6-102.0s)
  ASR segments: 40 (135.4-138.9s)
  ASR segments: 50 (156.4-158.0s)
  ASR segments: 60 (177.3-179.8s)
  ASR segments: 70 (208.4-210.7s)
  ASR segments: 80 (247.0-252.0s)
  ASR segments: 90 (279.0-280.5s)
    WER 27.5% | CER 16.5% | TermPresence 0.0% | NSHR 0.000 | RTF 0.07 | 22.0s
  CHI-003EC | hp
  ASR segments: 1 (4.0-6.9s)
  ASR segments: 10 (32.9-36.1s)
  ASR segments: 20 (67.0-69.6s)
  ASR segments: 30 (97.6-102.0s)
  ASR segments: 40 (144.6-149.8s)
  ASR segments: 50 (189.0-191.0s)
  ASR segments: 60 (236.3-239.3s)
  ASR segments: 70 (280.6-285.1s)
    WER 29.5% | CER 16.3% | TermPresence 0.0% | NSHR 0.000 | RTF 0.10 | 30.0s
{
  "records": 2,
  "summaries": 2,
  "out": "/kaggle/working/eval_smoke_tier2"
}


In [33]:
import json
from pathlib import Path

t2 = json.loads(Path("/kaggle/working/eval_smoke_tier2/tier2_m3av_results.json").read_text())
print("SMOKE tier2 records:", len(t2))
r2 = t2[0]
print(f"SMOKE tier2 WER: {r2["wer"]:.2%} | RTF: {r2["rtf"]:.3f} | NSHR: {r2["nshr"]}")


SMOKE tier2 records: 2
SMOKE tier2 WER: 27.55% | RTF: 0.072 | NSHR: 0.0


## Full run  Tier 2 (M3AV), batched per duration bucket

In [47]:
import ctypes, glob, os, site

# Locate the pip-installed NVIDIA libs and load them into the process
# (setting LD_LIBRARY_PATH after Python starts does NOT work, so preload instead)
sp = site.getsitepackages()[0]
for pattern in [
    "nvidia/cublas/lib/libcublas.so.12*",
    "nvidia/cublas/lib/libcublasLt.so.12*",
    "nvidia/cudnn/lib/libcudnn*.so.9*",
]:
    for lib in sorted(glob.glob(os.path.join(sp, pattern))):
        try:
            ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)
        except OSError as e:
            print("skip", lib, e)

# sanity check
ctypes.CDLL("libcublas.so.12")
print("cuBLAS 12 loaded")

cuBLAS 12 loaded


In [ ]:
import os
import shutil
import sys
import zipfile
from pathlib import Path
from types import SimpleNamespace

BACKEND_DIR = os.environ.get("BACKEND_DIR", os.path.join("/kaggle/working", "finalproject_cpe027", "backend"))
os.chdir(BACKEND_DIR)
sys.path.insert(0, BACKEND_DIR)
from app import eval_data, eval_runner

INPUT = Path("/kaggle/input")
raw_roots = sorted({
    str(p.parent.parent) for p in INPUT.rglob("extracted")
    if p.is_dir() and p.parent.name == "M3AV"
})
env_roots = [r for r in os.environ.get("M3AV_CLIP_ROOT", "").split(chr(10)) if r.strip()]
m3av_zips = [z for z in os.environ.get("M3AV_ZIPS", "").split(chr(10)) if z.strip()]

if raw_roots:
    inputs = raw_roots
elif env_roots:
    inputs = env_roots
elif m3av_zips:
    inputs = m3av_zips
else:
    raise RuntimeError("No M3AV splits are attached; Tier 2 cannot run.")

CONDITIONS = "arm_a,hp,hp+sg,hp_ffmpeg,hp_wiener,hp_wiener_oa"
BUCKETS = ["<5m", "5-10m", "10-20m", ">=20m"]
MOUNTS = Path("/kaggle/working/mounts")

def extracted_root(zip_path):
    dest = MOUNTS / ("m3av_" + Path(zip_path).stem)
    marker = dest / "M3AV" / "extracted"
    if not marker.is_dir():
        if dest.exists():
            shutil.rmtree(dest)
        dest.mkdir(parents=True, exist_ok=True)
        print("Extracting", Path(zip_path).name, flush=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(dest)
    return dest / "M3AV" / "extracted"

for src in inputs:
    if str(src).endswith(".zip"):
        clip_root = extracted_root(src)
        zip_source = src
    else:
        clip_root = Path(src)
        zip_source = None
        if not clip_root.is_dir():
            print("Skip missing root:", clip_root)
            continue

    for bucket in BUCKETS:
        selected = [c for c in eval_data.load_m3av_clips(str(clip_root))
                    if eval_data.duration_bucket(c.duration_seconds) == bucket]
        if not selected:
            continue
        out_name = Path(src).stem if str(src).endswith(".zip") else clip_root.name
        out_dir = "/kaggle/working/batches/tier2_" + \
            bucket.replace("<", "lt_").replace(">", "gt_").replace("+", "p") + "_" + out_name
        orig_loader = eval_runner.load_m3av_clips
        eval_runner.load_m3av_clips = lambda data_root=None, _sel=selected: _sel
        args = SimpleNamespace(
            tier="2",
            conditions=CONDITIONS,
            m3av_root=str(clip_root),
            data_root=None,
            out=out_dir,
            limit=None,
            resume=True,
        )
        try:
            records, _summary = eval_runner.run_tier2(args)
        finally:
            eval_runner.load_m3av_clips = orig_loader
        print(
            "tier2 %s bucket %s: %d records -> %s"
            % (clip_root.name, bucket, len(records), out_dir)
        )

    if zip_source is not None:
        shutil.rmtree(clip_root.parent.parent, ignore_errors=True)
        print("Cleaned", clip_root.parent.parent, flush=True)


Loading ASR model...
Tier 2: 63 clips
  CHI-004BD | arm_a
  ASR segments: 1 (4.9-16.2s)
  ASR segments: 10 (53.1-59.0s)
  ASR segments: 20 (88.9-91.9s)
  ASR segments: 30 (122.0-123.9s)
  ASR segments: 40 (169.8-174.6s)
  ASR segments: 50 (217.9-223.2s)
  ASR segments: 60 (264.3-266.1s)
  ASR segments: 70 (291.3-295.1s)
    WER 2.3% | CER 0.8% | TermPresence 0.0% | NSHR 0.000 | RTF 0.07 | 20.1s
  CHI-004BD | hp
  ASR segments: 1 (5.7-16.2s)
  ASR segments: 10 (53.1-59.0s)
  ASR segments: 20 (88.9-91.9s)
  ASR segments: 30 (125.4-130.8s)
  ASR segments: 40 (179.5-185.3s)
  ASR segments: 50 (229.4-235.3s)
  ASR segments: 60 (281.3-286.7s)
    WER 2.0% | CER 0.7% | TermPresence 0.0% | NSHR 0.000 | RTF 0.09 | 28.0s
  CHI-004BD | hp+sg
  ASR segments: 1 (5.7-16.2s)
  ASR segments: 10 (53.1-59.0s)
  ASR segments: 20 (88.9-91.9s)
  ASR segments: 30 (125.4-130.8s)
  ASR segments: 40 (179.5-185.3s)
  ASR segments: 50 (229.4-235.3s)
  ASR segments: 60 (281.3-286.7s)
    WER 2.0% | CER 0.7% | Ter

## Merge batch outputs into final results

In [ ]:
import os
from pathlib import Path

REPO_DIR = os.environ.get("REPO_DIR", "/kaggle/working/finalproject_cpe027")
MERGE_SCRIPT = Path(REPO_DIR) / "kaggle" / "merge_eval_results.py"
if not MERGE_SCRIPT.is_file():
    MERGE_SCRIPT = Path(REPO_DIR) / "merge_eval_results.py"
os.chdir(REPO_DIR)

!python {MERGE_SCRIPT} --batch-dir /kaggle/working/batches --out /kaggle/working/final


In [ ]:
from pathlib import Path

import json

names = [
    "tier1_acl_results.json",
    "tier1_acl_results.csv",
    "tier1_acl_summary.json",
    "tier2_m3av_results.json",
    "tier2_m3av_results.csv",
    "tier2_m3av_summary.json",
]
final = Path("/kaggle/working/final")
for name in names:
    p = final / name
    print(name, "OK" if p.is_file() else f"MISSING: {p}")

t1_path = final / "tier1_acl_results.json"
t2_path = final / "tier2_m3av_results.json"
if not t1_path.is_file() or not t2_path.is_file():
    raise RuntimeError("Final tier JSON files not found; merge step may have failed")
t1 = json.loads(t1_path.read_text(encoding="utf-8"))
t2 = json.loads(t2_path.read_text(encoding="utf-8"))
print("Tier 1 records:", len(t1))
print("Tier 2 records:", len(t2))
if len(t1) == 0:
    raise RuntimeError("Final Tier 1 JSON has 0 records")
if len(t2) == 0:
    raise RuntimeError("Final Tier 2 JSON has 0 records")
